In [3]:
import os
from difflib import SequenceMatcher
from pathlib import Path

import cv2
import numpy as np
from paddleocr import PaddleOCR

c:\Projects\data-analysis\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [4]:
ocr = PaddleOCR(use_angle_cls=True, enable_mkldnn=False, lang='ru', ocr_version='PP-OCRv5')

C:\Users\Sergey\AppData\Local\Temp\ipykernel_31692\3947172038.py:1: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=True, enable_mkldnn=False, lang='ru', ocr_version='PP-OCRv5')
c:\Projects\data-analysis\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `C:\Users\Sergey\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

inference.yml:   0%|          | 0.00/766 [00:00<?, ?B/s]

...

inference.pdiparams:   0%|          | 0.00/7.81M [00:00<?, ?B/s]

inference.yml: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

inference.json: 0.00B [00:00, ?B/s]

In [8]:

images_dir = Path(r"..\..\Титры Лунтика Новый 8K Смотреть На ПК и не только")

SIMILARITY_THRESHOLD = 0.85

LATIN_TO_CYR = str.maketrans("aeopcxylmkivusnrABEHKMOPCTXYLIVUSNR", "аеорсхулмкивиснрАВЕНКМОРСТХУЛИВИСНР")

def normalize(text):
    return text.translate(LATIN_TO_CYR).lower()

prev_text = ""
unique_subtitles = []

for filename in sorted(images_dir.rglob('*'), key=lambda x: (len(str(x)), str(x))):
    if filename.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
        continue

    with open(filename, 'rb') as f:
        img = np.asarray(bytearray(f.read()), dtype=np.uint8)
        img = cv2.imdecode(img, cv2.IMREAD_COLOR)

    print(f"Обработка: {filename.name}")

    rec_texts = ocr.predict(img)[0]['rec_texts']
    text = " ".join(rec_texts).strip()

    if not text:
        continue

    norm_text = normalize(text)

    # Пропустить кадр, если он слишком похож на предыдущий
    if SequenceMatcher(None, normalize(prev_text), norm_text).ratio() >= SIMILARITY_THRESHOLD:
        print(f"  (пропущено как дубль)")
        continue

    print(f"Субтитры: {norm_text}")
    unique_subtitles.append(norm_text)
    prev_text = norm_text

print(f"\nУникальных субтитров: {len(unique_subtitles)}")


Обработка: frame_1.jpg
Субтитры: роли озвучивали
Обработка: frame_2.jpg
Субтитры: роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман
Обработка: frame_3.jpg
  (пропущено как дубль)
Обработка: frame_4.jpg
Субтитры: роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман автор сценария фёдор дмитриев
Обработка: frame_5.jpg
  (пропущено как дубль)
Обработка: frame_6.jpg
  (пропущено как дубль)
Обработка: frame_7.jpg
Субтитры: роли озвучивали олег куликович екатерина гороховская черняк елена шульман михаил автор сценария фёдор дмитриев режиссер сериала елена галдобина
Обработка: frame_8.jpg
Субтитры: роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман автор сценария фёдор дмитриев режиссер сериала елена галдобина режиссер серуу екатерина салабай
Обработка: frame_9.jpg
  (пропущено как дубль)
Обработка: frame_10.jpg
Субтитры: слена шульман автор сценария фёдор дмитриев режиссер сериала елена галдобина режиссе

KeyboardInterrupt: 

In [10]:
unique_subtitles = [
    'роли озвучивали',
    'роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман',
    'роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман автор сценария фёдор дмитриев',
    'роли озвучивали олег куликович екатерина гороховская черняк елена шульман михаил автор сценария фёдор дмитриев режиссер сериала елена галдобина',
    'роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман автор сценария фёдор дмитриев режиссер сериала елена галдобина режиссер серуу екатерина салабай',
    'слена шульман автор сценария фёдор дмитриев режиссер сериала елена галдобина режиссер серуу екатерина салабай художник-постановщик марина комаркевич'
]

In [17]:
from difflib import SequenceMatcher


def words_similar(a, b, threshold=0.7):
    if a == b:
        return True
    return SequenceMatcher(None, a, b).ratio() >= threshold


def find_new_tail(merged_words, new_words, threshold=0.7):
    """
    Ищем в new_words позицию, начиная с которой идёт НОВЫЙ текст,
    которого ещё нет в merged_words.
    
    Идём с конца new_words и ищем последнее слово, которое похоже
    на последнее слово merged. Всё что после — новое.
    """
    if not merged_words or not new_words:
        return new_words

    last_merged = merged_words[-1]

    best_pos = -1
    for j in range(len(new_words)):
        if words_similar(new_words[j], last_merged, threshold):
            # Проверяем контекст: несколько слов перед этой позицией
            # тоже должны совпадать с концом merged
            context_len = min(3, j + 1, len(merged_words))
            if context_len <= 1:
                best_pos = j
                continue

            matches = 0
            for k in range(context_len):
                m_word = merged_words[-(context_len - k)]
                n_word = new_words[j - (context_len - 1 - k)]
                if words_similar(m_word, n_word, threshold):
                    matches += 1

            if matches / context_len >= 0.6:
                best_pos = j

    if best_pos >= 0 and best_pos + 1 < len(new_words):
        return new_words[best_pos + 1:]
    
    return None


def merge_scrolling_subtitles(subtitles, threshold=0.7):
    if not subtitles:
        return ""

    merged_words = subtitles[0].split()

    for i in range(1, len(subtitles)):
        new_words = subtitles[i].split()
        if not new_words:
            continue

        # Проверяем: может новый кадр целиком содержится в merged
        new_text = " ".join(new_words)
        merged_text = " ".join(merged_words)
        if new_text in merged_text:
            continue

        tail = find_new_tail(merged_words, new_words, threshold)

        if tail is not None and len(tail) > 0:
            merged_words.extend(tail)
        elif tail is None:
            merged_words.extend(new_words)

    return " ".join(merged_words)


In [19]:
unique_subtitles = [
    'роли озвучивали',
    'роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман',
    'роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман автор сценария фёдор дмитриев',
    'роли озвучивали олег куликович екатерина гороховская черняк елена шульман михаил автор сценария фёдор дмитриев режиссер сериала елена галдобина',
    'роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман автор сценария фёдор дмитриев режиссер сериала елена галдобина режиссер серуу екатерина салабай',
    'слена шульман автор сценария фёдор дмитриев режиссер сериала елена галдобина режиссер серуу екатерина салабай художник-постановщик марина комаркевич'
]

merge_scrolling_subtitles(unique_subtitles)

'роли озвучивали екатерина гороховская олег куликович михаил черняк елена шульман автор сценария фёдор дмитриев режиссер сериала елена галдобина режиссер серуу екатерина салабай художник-постановщик марина комаркевич'